# Notebook 1: Emotion CSV Inter-Rater Agreement Analysis

The [Jira Emotion dataset](https://web.archive.org/web/20210224080543/http://ansymore.uantwerpen.be/system/files/uploads/artefacts/alessandro/MSR16/archive3.zip) contains manually labeled ~2,000 Jira issue comments and ~4,000 sentences written by developers with emotions such as love, joy, surprise, anger, sadness and fear. It was published by Ortu et al. 2016 in ["The Emotional Side of Software Developers in JIRA"](https://ieeexplore.ieee.org/document/7832930).

The problem with this dataset is that it contains no contextual information about a comment, such as where the comment came from, who wrote it, or when it was written.

To fix that, we're going to load the Jira Emotion CSVs into the Jira PostgreSQL 2016 database dump (`emotion_dataset_jira.sql.tar.gz`). This database dump contains the contextual project data from Jira (e.g., author, timestamp, etc.). By joining the Jira Emotion CSVs to this database, we recover the context that is missing from the CSVs. Once both datasets are in the same database, we can JOIN them together using the `issue_key` and `timestamp`.

This is the first step in a pipeline that ends with emotion-labeled comment data that Kaiaulu can use for analysis.

### Notebook Overview

This notebook analyzes the inter-rater agreement of the manually labeled emotion CSV files from the Jira emotion dataset. The goal is to understand how consistently the human raters agreed on the presence of emotions in Jira issue comments and sentences, and to identify which comment IDs are the most reliably labeled before joining them to the Jira SQL database.

The emotion-labeled data comes from three groups:

- **Group 1**: ~392 issue comments labeled across 6 emotions by 16 raters (4 raters per comment)
- **Group 2**: ~1,600 issue comments labeled across love, joy, and sadness by 3 raters (same 3 raters per comment)
- **Group 3**: ~4,000 issue sentences labeled across anger, joy, love, and sadness by 3 raters at the sentence level

Groups 1 and 2 data originate from the Murgia et al. 2014 ["Do Developers Feel Emotions?
An Exploratory Analysis of Emotions in Software Artifacts"](https://dl.acm.org/doi/pdf/10.1145/2597073.2597086) labeling experiments.

Group 3 originates from Ortu et al.'s 2016 ["The Emotional Side of Software Developers in JIRA"](https://ieeexplore.ieee.org/document/7832930) work on sentence-level labeling. This paper also packaged all three groups together into the dataset archive we download.



### What is inter-rater agreement?

Inter-rater agreement measures how consistently multiple raters labeled the same comment. In this dataset, each comment (or sentence) was labeled by multiple raters who independently marked which emotions they detected. For each emotion, we calculate the proportion of raters who agreed it was present as a decimal between 0 and 1.

- `0.0` means none of the raters marked that emotion
- `1.0` means all raters agreed the emotion was present
- `0.5` means half the raters marked it present

Comments with higher agreement scores are more reliably labeled and are better candidates for joining to the Jira database.

### Step 1: Install and import dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

### Step 2: Set data paths

Update these paths to point to where you saved the dataset archive on your local computer.

In [ ]:
GROUP1_PATH = os.path.expanduser('~/PATH_TO/archive3/Groups/group1/')
GROUP2_PATH = os.path.expanduser('~/PATH_TO/archive3/Groups/group2/')
GROUP3_PATH = os.path.expanduser('~/PATH_TO/archive3/Groups/group3/')

---
## Helper Functions

The following functions are reused across all three groups to compute inter-rater agreement tables and summary statistics.

- `compute_agreement_table`: takes a list of rater dataframes and an emotion column list, and returns one table where each row is a comment and each emotion column is the proportion of raters who marked that emotion present.
- `summarize_agreement`: takes an agreement table and an aggregation function (e.g. `np.mean` or `np.std`) and returns a summary series across all comments per emotion.

In [ ]:
def compute_agreement_table(rater_dfs, emotion_cols, id_col='id', n_raters=None):
    """
    Given a list of rater dataframes, compute inter-rater agreement per comment.
    Each emotion column in the output contains the proportion of raters who marked
    that emotion present (count of 'x' / number of raters).

    Parameters:
        rater_dfs: list of DataFrames, one per rater
        emotion_cols: list of emotion column names to compute agreement for
        id_col: name of the comment ID column
        n_raters: total number of raters per comment (used as denominator)

    Returns:
        DataFrame with one row per comment and agreement scores per emotion
    """
    # Stack all rater dataframes
    combined = pd.concat(rater_dfs, ignore_index=True)

    # Convert 'x' to 1 and blank/NaN to 0 for each emotion column
    for col in emotion_cols:
        combined[col] = combined[col].apply(lambda v: 1 if str(v).strip().lower() == 'x' else 0)

    # Sum ratings per comment
    agreement = combined.groupby(id_col)[emotion_cols].sum().reset_index()

    # Divide by number of raters to get proportion
    if n_raters:
        agreement[emotion_cols] = agreement[emotion_cols] / n_raters

    return agreement


def summarize_agreement(agreement_df, emotion_cols, agg_func):
    """
    Aggregate all comments for each emotion column.

    Parameters:
        agreement_df: DataFrame output from compute_agreement_table
        emotion_cols: list of emotion column names
        agg_func: aggregation function to apply (e.g. np.mean or np.std)

    Returns:
        Series with the aggregated value per emotion
    """
    return agreement_df[emotion_cols].apply(agg_func)


def plot_agreement_histogram(agreement_df, emotion_cols, title, bins=10):
    """
    Plot a histogram of agreement score distribution for each emotion.

    Parameters:
        agreement_df: DataFrame output from compute_agreement_table
        emotion_cols: list of emotion column names
        title: plot title
        bins: number of histogram bins
    """
    n = len(emotion_cols)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=False)
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, emotion_cols):
        ax.hist(agreement_df[col], bins=bins, edgecolor='black')
        ax.set_title(col)
        ax.set_xlabel('Agreement Score')
        ax.set_ylabel('Number of Comments')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

---
## Group 1

Group 1 has 16 raters and each received a file with 98 comments to label. Each comment was assigned to exactly 4 out of the 16 raters, so each comment has 4 ratings total.

The dataset contains 16 CSV files (one per rater). Each file has:
- `id`: the Jira comment ID
- `love`, `joy`, `surprise`, `anger`, `sadness`, `fear`: marked with `x` if the rater identified that emotion, blank otherwise
- `notes`: optional rater notes
- `comment N`, `comment N-1`, ...: the target comment and its surrounding context (prior comments in the same Jira issue thread). Raters were only asked to label the emotions in `comment N`.

The extra context columns are not needed for analysis and will be dropped.

In [ ]:
GROUP1_EMOTIONS = ['love', 'joy', 'surprise', 'anger', 'sadness', 'fear']
GROUP1_N_RATERS = 4  # each comment was labeled by 4 out of 16 raters

# Load all 16 rater files
group1_files = sorted(glob.glob(os.path.join(GROUP1_PATH, 'Document*.csv')))
print(f'Found {len(group1_files)} rater files for Group 1')

group1_dfs = []
for f in group1_files:
    df = pd.read_csv(f, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    # Keep only id and emotion columns
    keep_cols = ['id'] + [c for c in GROUP1_EMOTIONS if c in df.columns]
    group1_dfs.append(df[keep_cols])

print('Sample from first rater file:')
group1_dfs[0].head(10)

In [ ]:
# Build inter-rater agreement table for Group 1
group1_agreement = compute_agreement_table(
    group1_dfs,
    emotion_cols=GROUP1_EMOTIONS,
    id_col='id',
    n_raters=GROUP1_N_RATERS
)

print(f'Group 1 agreement table: {group1_agreement.shape[0]} comments, {group1_agreement.shape[1]} columns')
group1_agreement.head(10)

In [ ]:
# Sanity check: no comment should have an agreement score above 1.0 
# (which would mean more than 4 raters labeled it)
violations = group1_agreement[GROUP1_EMOTIONS].gt(1.0).any(axis=1).sum()
print(f'Comments with agreement score > 1.0 (expected 0): {violations}')

In [ ]:
# Average agreement per emotion across all comments
print('Group 1 - Average inter-rater agreement per emotion:')
print(summarize_agreement(group1_agreement, GROUP1_EMOTIONS, np.mean).round(4))

print('\nGroup 1 - Standard deviation of inter-rater agreement per emotion:')
print(summarize_agreement(group1_agreement, GROUP1_EMOTIONS, np.std).round(4))

In [ ]:
# Histogram of agreement score distribution per emotion for Group 1
plot_agreement_histogram(
    group1_agreement,
    GROUP1_EMOTIONS,
    title='Group 1 — Inter-Rater Agreement Distribution',
    bins=5
)

---
## Group 2

Group 2 extends Group 1 but focuses only on the three emotions that showed fair agreement in the Murgia et al. 2014 study: love, joy, and sadness. 3 raters labeled 1,600 comments. The same 3 raters labeled every comment.

The dataset contains 3 CSV files (one per rater). Each file has:
- `id`: the Jira comment ID
- `love`, `joy`, `sadness`: the three labeled emotions, marked with `x` if present
- `neutral`: marked with `x` if none of the three emotions are present
- `surprise`, `anger`, `fear`: present in the CSV files but not part of the Group 2 labeling scheme (confirmed empty in the sanity check below)
- `comment`: the comment text

Although all emotion columns exist, the paper states only love, joy, sadness, and neutral were used. We will sanity check that anger, fear, and surprise are all empty before proceeding.

In [ ]:
GROUP2_EMOTIONS = ['love', 'joy', 'sadness']  # only these were labeled
GROUP2_N_RATERS = 3

# Load all 3 rater files
group2_files = sorted(glob.glob(os.path.join(GROUP2_PATH, 'emoiton_*.csv')))
print(f'Found {len(group2_files)} rater files for Group 2')

group2_dfs = []
for f in group2_files:
    df = pd.read_csv(f, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    group2_dfs.append(df)

print('Sample from first rater file:')
group2_dfs[0].head(10)

In [ ]:
# Sanity check: anger, fear, and surprise should be completely empty across all 3 files
unexpected_cols = ['anger', 'fear', 'surprise']
for i, df in enumerate(group2_dfs):
    for col in unexpected_cols:
        if col in df.columns:
            non_empty = df[col].apply(lambda v: str(v).strip().lower() == 'x').sum()
            if non_empty > 0:
                print(f'WARNING: rater file {i+1} has {non_empty} non-empty values in "{col}"')
            else:
                print(f'OK: rater file {i+1} — "{col}" is empty')

In [ ]:
# Keep only id, comment, and the three labeled emotions
for i in range(len(group2_dfs)):
    keep = ['id'] + [c for c in GROUP2_EMOTIONS if c in group2_dfs[i].columns]
    group2_dfs[i] = group2_dfs[i][keep]

# Build inter-rater agreement table for Group 2
group2_agreement = compute_agreement_table(
    group2_dfs,
    emotion_cols=GROUP2_EMOTIONS,
    id_col='id',
    n_raters=GROUP2_N_RATERS
)

print(f'Group 2 agreement table shape: {group2_agreement.shape}')
group2_agreement.head(10)

In [ ]:
# Sanity check: no agreement score above 1.0
violations = group2_agreement[GROUP2_EMOTIONS].gt(1.0).any(axis=1).sum()
print(f'Comments with agreement score > 1.0 (expected 0): {violations}')

In [ ]:
print('Group 2 - Average inter-rater agreement per emotion:')
print(summarize_agreement(group2_agreement, GROUP2_EMOTIONS, np.mean).round(4))

print('\nGroup 2 - Standard deviation of inter-rater agreement per emotion:')
print(summarize_agreement(group2_agreement, GROUP2_EMOTIONS, np.std).round(4))

In [ ]:
plot_agreement_histogram(
    group2_agreement,
    GROUP2_EMOTIONS,
    title='Group 2 - Inter-Rater Agreement Distribution',
    bins=4
)

---
## Group 3

Group 3 contains sentence level Jira comments. Each comment was split into individual sentences, and each sentence was labeled separately.

The dataset contains 4 CSV files, one per emotion (anger, joy, love, sadness). Each file has:
- `id`: a sentence-level ID in the format `commentID_sentenceNumber` (e.g. `1001112_1`)
- `love`, `joy`, `surprise`, `anger`, `sadness`, `fear`: marked with `x` if present
- `notes`, `probability`: additional columns
- `comment`: the sentence text

Because `jira_issue_comment` stores data at the comment level, we will first build a sentence-level agreement table, then group sentences by their comment ID to get a comment-level table.

In [ ]:
GROUP3_EMOTIONS = ['love', 'joy', 'anger', 'sadness']  # only these were labeled
GROUP3_N_RATERS = 3

# Load all 4 emotion files
group3_files = sorted(glob.glob(os.path.join(GROUP3_PATH, 'SentenceWithEmotion_TRAINING_*.csv')))
print(f'Found {len(group3_files)} files for Group 3')

group3_dfs = []
for f in group3_files:
    df = pd.read_csv(f, dtype=str)
    df.columns = [c.strip() for c in df.columns]
    keep = ['id'] + [c for c in GROUP3_EMOTIONS if c in df.columns]
    group3_dfs.append(df[keep])
    print(f'{os.path.basename(f)}: {len(df)} rows')

print('\nSample from ANGER file:')
group3_dfs[0].head(10)

In [ ]:
# Combine all 4 files into one sentence-level dataframe
group3_all = pd.concat(group3_dfs, ignore_index=True)

# Convert 'x' to 1 and blank/NaN to 0
for col in GROUP3_EMOTIONS:
    group3_all[col] = group3_all[col].apply(lambda v: 1 if str(v).strip().lower() == 'x' else 0)

# Build sentence-level agreement table
# Each sentence appears once per file it was labeled in
# Sum per sentence ID, then divide by n_raters
group3_sentence = group3_all.groupby('id')[GROUP3_EMOTIONS].sum().reset_index()
group3_sentence[GROUP3_EMOTIONS] = group3_sentence[GROUP3_EMOTIONS] / GROUP3_N_RATERS

print(f'Group 3 sentence-level agreement table shape: {group3_sentence.shape}')
group3_sentence.head(10)

In [ ]:
print('Group 3 — Average inter-rater agreement per emotion (sentence level):')
print(summarize_agreement(group3_sentence, GROUP3_EMOTIONS, np.mean).round(4))

print('\nGroup 3 — Standard deviation of inter-rater agreement per emotion (sentence level):')
print(summarize_agreement(group3_sentence, GROUP3_EMOTIONS, np.std).round(4))

In [ ]:
# Roll up from sentence level to comment level
# Extract comment ID from sentence ID (e.g. '1001112_1' -> '1001112')
group3_sentence['comment_id'] = group3_sentence['id'].apply(lambda x: str(x).split('_')[0])

# Average agreement scores across all sentences within the same comment
group3_comment = group3_sentence.groupby('comment_id')[GROUP3_EMOTIONS].mean().reset_index()
group3_comment.rename(columns={'comment_id': 'id'}, inplace=True)

print(f'Group 3 comment-level agreement table shape: {group3_comment.shape}')
group3_comment.head(10)

In [ ]:
print('Group 3 - Average inter-rater agreement per emotion (comment level):')
print(summarize_agreement(group3_comment, GROUP3_EMOTIONS, np.mean).round(4))

print('\nGroup 3 - Standard deviation of inter-rater agreement per emotion (comment level):')
print(summarize_agreement(group3_comment, GROUP3_EMOTIONS, np.std).round(4))

In [ ]:
plot_agreement_histogram(
    group3_comment,
    GROUP3_EMOTIONS,
    title='Group 3 - Inter-Rater Agreement Distribution (Comment Level)',
    bins=5
)

---
## Cross-Group Comparison

Now that we've got the inter-rater agreement distribution from all 3 groups, we check if any comment IDs appear in more than one group. If they do, we can compare how consistently they were labeled across groups, even though the number of raters and the labeling method (comment-level vs. sentence-level) differ.

We only compare love, joy, and sadness since those are the only emotions present in all three groups.

In [ ]:
CROSS_EMOTIONS = ['love', 'joy', 'sadness']

# Get comment ID sets from each group
g1_ids = set(group1_agreement['id'].astype(str))
g2_ids = set(group2_agreement['id'].astype(str))
g3_ids = set(group3_comment['id'].astype(str))

shared_1_2 = g1_ids & g2_ids
shared_1_3 = g1_ids & g3_ids
shared_2_3 = g2_ids & g3_ids
shared_all = g1_ids & g2_ids & g3_ids

print(f'Comment IDs shared between Group 1 and Group 2: {len(shared_1_2)}')
print(f'Comment IDs shared between Group 1 and Group 3: {len(shared_1_3)}')
print(f'Comment IDs shared between Group 2 and Group 3: {len(shared_2_3)}')
print(f'Comment IDs shared across all three groups: {len(shared_all)}')

In [ ]:
# Build cross-group comparison table for all shared IDs
all_shared_ids = shared_1_2 | shared_1_3 | shared_2_3

if len(all_shared_ids) > 0:
    # Prepare each group's subset
    g2_sub = group2_agreement[group2_agreement['id'].astype(str).isin(all_shared_ids)][['id'] + CROSS_EMOTIONS].copy()
    g2_sub.columns = ['id'] + [f'g2_{e}' for e in CROSS_EMOTIONS]

    g3_sub = group3_comment[group3_comment['id'].astype(str).isin(all_shared_ids)][['id'] + [e for e in CROSS_EMOTIONS if e in GROUP3_EMOTIONS]].copy()
    g3_sub.columns = ['id'] + [f'g3_{e}' for e in CROSS_EMOTIONS if e in GROUP3_EMOTIONS]

    # Merge all
    cross = g2_sub.merge(g3_sub, on='id', how='outer')
    print(f'Cross-group comparison table shape: {cross.shape}')
    display(cross)
else:
    print('No comment IDs are shared across groups.')


---
## Selecting Reliable Comment IDs

From the agreement tables and histograms above, we can now identify which comment IDs are the most reliably labeled. A reliable comment is one where the average agreement score is high (raters consistently agreed) and the standard deviation is low (scores are consistent, not spread out)

The cell below lets you set your own threshold. Any comment with an average agreement score at or above the threshold across at least one emotion will be included. You can adjust `AGREEMENT_THRESHOLD` based on what you see in the histograms above.

These selected comment IDs will be used in a later notebook to join the emotion labels to the Jira SQL database.

In [ ]:
# Adjust this threshold based on the histograms above
AGREEMENT_THRESHOLD = 0.75

# Select comment IDs from each group that meet the threshold for at least one emotion
g1_reliable = group1_agreement[group1_agreement[GROUP1_EMOTIONS].max(axis=1) >= AGREEMENT_THRESHOLD]['id']
g2_reliable = group2_agreement[group2_agreement[GROUP2_EMOTIONS].max(axis=1) >= AGREEMENT_THRESHOLD]['id']
g3_reliable = group3_comment[group3_comment[GROUP3_EMOTIONS].max(axis=1) >= AGREEMENT_THRESHOLD]['id']

print(f'Group 1 reliable comment IDs (agreement >= {AGREEMENT_THRESHOLD}): {len(g1_reliable)}')
print(f'Group 2 reliable comment IDs (agreement >= {AGREEMENT_THRESHOLD}): {len(g2_reliable)}')
print(f'Group 3 reliable comment IDs (agreement >= {AGREEMENT_THRESHOLD}): {len(g3_reliable)}')

---
## How many comments survive each threshold?

Setting a threshold tells you how strict your quality bar is, but it doesn't tell you how many comments actually pass it. If a threshold is too high, you might end up with almost no data to work with, which defeats the purpose.

The table below counts how many comments meet each meaningful agreement level for each group and emotion. The levels shown are the only ones that can actually occur. With 4 raters (Group 1), agreement can only be 0, 0.25, 0.50, 0.75, or 1.0. With 3 raters (Groups 2 and 3), it can only be 0, 0.33, 0.67, or 1.0. Use these counts to pick a threshold that keeps enough comments without sacrificing too much quality.

In [ ]:
# Group 1: 4 raters per comment — thresholds at 1/4, 2/4, 3/4, 4/4
g1_levels = [('1 / 4 raters (>= 0.25)', 1/4),
             ('2 / 4 raters (>= 0.50)', 2/4),
             ('3 / 4 raters (>= 0.75)', 3/4),
             ('4 / 4 raters (= 1.00)',  1.00)]

# Group 2 & 3: 3 raters per comment — thresholds at 1/3, 2/3, 3/3
g23_levels = [('1 / 3 raters (>= 0.33)', 1/3),
              ('2 / 3 raters (>= 0.67)', 2/3),
              ('3 / 3 raters (= 1.00)',  1.00)]

def count_table(agreement_df, emotion_cols, levels, total_comments):
    rows = []
    for label, t in levels:
        row = {'threshold': label}
        for col in emotion_cols:
            row[col] = int((agreement_df[col] >= t).sum())
        row['any emotion'] = int((agreement_df[emotion_cols].max(axis=1) >= t).sum())
        rows.append(row)
    df = pd.DataFrame(rows).set_index('threshold')
    df.loc['total comments'] = total_comments
    return df

print('Group 1 — Absolute comment counts at each agreement level (out of 392 total):')
display(count_table(group1_agreement, GROUP1_EMOTIONS, g1_levels, len(group1_agreement)))

print('\nGroup 2 — Absolute comment counts at each agreement level (out of 1600 total):')
display(count_table(group2_agreement, GROUP2_EMOTIONS, g23_levels, len(group2_agreement)))

print('\nGroup 3 — Absolute comment counts at each agreement level (out of 2297 total, comment level):')
display(count_table(group3_comment, GROUP3_EMOTIONS, g23_levels, len(group3_comment)))